# Stretch: more cleaning, more charts

For when the live exercise is done and there is time left. These get
harder as you go, so nobody is expected to finish them all.

The rules from the live exercise still hold: every answer is a function
that **returns**, or a chart with a title that gives the answer. Where a
stretch has `assert` lines, uncomment them once your code exists: they are
the real numbers, so they tell you when you are right.

Each stretch adds one new idea, and says what it is when you get there.

## Start here

Run this cell. It loads the raw messy file, the cleaned version, and the
clean file, using the functions from the live exercise.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

sys.path.insert(0, "sessions/session-09/solutions")
from cleaning import clean_plays, load_messy

raw = load_messy("data/messy/plays_messy.csv")
plays = clean_plays(raw)
clean = pd.read_csv("data/clean/plays.csv")

os.makedirs("output", exist_ok=True)
print(len(raw), len(plays), len(clean))

## Stretch 1: a damage report

In notebook 01 you found the damage one column at a time. Write
`damage_report(df)`, which does it for every column at once and returns a
DataFrame with one row per column:

| column | blanks | spellings | after strip and lower |
|---|---|---|---|
| genre | 36 | 33 | 9 |

* `blanks`: how many values are the empty string `""`
* `spellings`: how many different values there are
* `after strip and lower`: how many different values are left after
  `.str.strip().str.lower()`

**New idea:** you build the report as a list of dictionaries, one per
column, and hand it to `pd.DataFrame`. Session 4 promised that a DataFrame
is a list of dicts with a faster engine; this is that promise, used
backwards.

Every column of `raw` holds text except `play_id` and `skipped`. Use
`.astype(str)` on each column first, so the same code works on both.

In [ ]:
# TODO: def damage_report(df):


# report = damage_report(raw)
# assert len(report) == 9
# assert report.set_index("column").loc["device", "spellings"] == 21
# assert report.set_index("column").loc["device", "after strip and lower"] == 6

Which column has the most spellings, and why is that one not damage at
all? Which column has a different number of `blanks` from what
`isna().sum()` reported on the default read in notebook 01, and why?

*Your answer here.*

## Stretch 2: draw the wrong dates

Notebook 01 counted 159 dates that `format="mixed"` gets wrong. A count is
easy to ignore. Draw it.

Make **one** line chart of plays per month with **two** lines: months from
the deliberate parse (use `plays`), and months from `format="mixed"` on
`raw.drop_duplicates()`.

**New idea:** call `ax.plot` twice on the same `ax`, give each a
`label="..."`, and call `ax.legend()`. Two lines means a legend is needed;
one line never does.

In [ ]:
# TODO

Which month moved most, and in which direction? Could you have spotted the
problem from the wrong line alone, without the right one next to it?

*Your answer here.*

## Stretch 3: what were the unknowns?

69 plays became `"unknown"`. The clean file knows what they really were.
Put the two side by side with `merge`, as in session 8.

**New idea:** `suffixes=("_ours", "_clean")` names the columns that appear
on both sides, and `validate="one_to_one"` makes pandas stop if a
`play_id` turns up twice on either side. Session 8 used `"many_to_one"`;
here each play exists once in each table.

In [ ]:
# TODO: merged = plays.merge(clean, on="play_id", ...)


# assert len(merged) == 2183
# unknowns = merged[merged["device_ours"] == "unknown"]
# assert len(unknowns) == 69

Count what the unknowns really were. Compare that with the share of each
device in the whole log (`value_counts(normalize=True)`).

Were the blanks spread across devices about evenly, or did one device lose
far more than its share? Why does that decide whether dropping them would
have been safe?

In [ ]:
# TODO

*Your answer here.*

## Stretch 4: an honest skip-rate chart

The homework found that speaker is skipped most, and that `"unknown"`
should not win the ranking. Draw the chart that says so.

* horizontal bars (`ax.barh`), sorted so the highest is at the top
* every bar labelled with the rate **and** the number of plays, like
  `14.1% of 326`
* `"unknown"` drawn in grey, so it is visibly not a device
* a title that gives the answer

**New idea:** `ax.bar_label(bars, labels=[...])` takes a list of your own
text, one per bar, instead of a format. Build the list with an f-string in
a loop or a list comprehension. And `color=` accepts a list of colours too,
one per bar.

In [ ]:
# TODO

## Stretch 5: two years, side by side

Session 4's stretch 5 built a genre-by-year table by hand, and found that
Folk fell most. Draw it: two bar charts next to each other, plays per genre
in 2024 on the left and 2025 on the right.

**New idea:** `fig, (left, right) = plt.subplots(1, 2, sharey=True)`.
`sharey=True` gives both charts the same y-axis. Without it, each chart
scales itself and two bars of different heights can look the same.

In [ ]:
# TODO


# genre_year = ...  a table with a row per genre and a column per year
# assert genre_year.loc["Folk", "2024"] - genre_year.loc["Folk", "2025"] == 34

Try it once without `sharey=True`. What does a reader believe from that
version?

*Your answer here.*

## Stretch 6: guess the missing minutes, and get marked

94 plays have no minutes. Notebook 01 left them as `NaN`, and the total
came out at 8,608.0 instead of the clean file's 8,980.4.

Filling a gap with a best guess is called **imputing**. Try four ways and
compare the totals with the truth:

1. leave them missing
2. fill with the average of all plays
3. fill with the median of **that artist's** plays
4. fill with the median of that artist's plays **with the same `skipped`
   value** (skipped plays are short)

**New idea:** `plays.groupby("artist_name")["minutes_played"].transform("median")`
returns a column as long as `plays`, where every row holds its own artist's
median. That lines up with `minutes_played` row for row, so it can go
straight into `fillna`. `groupby` with a list of two columns works the same
way.

`imputed_total(minutes)` returns the total of a filled-in column. Then make
a small table: method, total, and the difference from 8,980.4.

In [ ]:
# TODO


# assert round(plays["minutes_played"].sum(), 1) == 8608.0

**Think about it:**

1. Method 3 is worse than method 2. Using more information made the guess
   worse. Why? (Look at `plays.loc[plays["minutes_played"].isna(), "skipped"].mean()`.)
2. Method 1 has the biggest error by far. Is it still the most honest?
   What would you write next to the total under each method?
3. In real work there is no clean file to mark your guess. How would you
   choose, then?

*Your answers here.*